In [5]:
import numpy as np
import os
from skimage.draw import ellipse
from skimage.util import random_noise
from skimage.filters import gaussian
from skimage.morphology import dilation, disk
import imageio.v2 as imageio
import tifffile
import pandas as pd

In [ ]:
np.random.seed(42)

OUTDIR = "../Images/synthetic_images"
os.makedirs(OUTDIR, exist_ok=True)

n_images = 100
img_size = (512, 512)
nuclei_per_image = (50, 120)

def make_background(shape, amplitude=0.15):
    yy, xx = np.indices(shape)
    grad = (xx / shape[1] + yy / shape[0]) * 0.5
    return grad * amplitude

meta = []

for i in range(n_images):

    cond = "treatment" if i >= n_images//2 else "control"

    nuc_img = np.zeros(img_size, dtype=float)
    mem_img = np.zeros(img_size, dtype=float)
    gt = np.zeros(img_size, dtype=np.int32)

    nuc_img += make_background(img_size)

    n_nucs = np.random.randint(*nuclei_per_image)

    for label_id in range(1, n_nucs + 1):

        r1 = np.random.randint(6, 18)
        r2 = np.random.randint(6, 18)
        angle = np.random.rand() * 2*np.pi

        cy = np.random.randint(20, img_size[0]-20)
        cx = np.random.randint(20, img_size[1]-20)

        base_int = np.random.normal(0.6, 0.15)

        if cond == "treatment":
            base_int *= 1.2
            r2 = int(r1 * np.random.uniform(0.9, 1.05))

        rr, cc = ellipse(cy, cx, r1, r2, rotation=angle, shape=img_size)

        nuc_img[rr, cc] += base_int
        gt[rr, cc] = label_id

        # Membrane ring for subset
        if np.random.rand() < 0.5:
            ring_mask = np.zeros_like(gt, dtype=bool)
            ring_mask[rr, cc] = True
            ring = dilation(ring_mask, disk(3)) ^ ring_mask
            mem_img[ring] += base_int * 0.5

    # Blur
    nuc_img = gaussian(nuc_img, sigma=1)
    mem_img = gaussian(mem_img, sigma=1)

    # Add noise
    nuc_img = random_noise(nuc_img, mode='poisson')
    nuc_img = random_noise(nuc_img, mode='gaussian', var=0.005)

    # Bleed-through
    mem_img += nuc_img * 0.05

    # Normalize to 8-bit
    nuc_img = (nuc_img - nuc_img.min()) / (nuc_img.max() - nuc_img.min())
    mem_img = (mem_img - mem_img.min()) / (mem_img.max() - mem_img.min())

    nuc_img = (nuc_img * 255).astype(np.uint8)
    mem_img = (mem_img * 255).astype(np.uint8)

    # Save files (INSIDE loop!)
    imageio.imwrite(os.path.join(OUTDIR, f"img_{i:03d}_ch0.png"), nuc_img)
    imageio.imwrite(os.path.join(OUTDIR, f"img_{i:03d}_ch1.png"), mem_img)
    tifffile.imwrite(os.path.join(OUTDIR, f"img_{i:03d}_gt.tif"), gt)

    meta.append({
        "image": f"img_{i:03d}",
        "condition": cond,
        "n_true_objects": n_nucs
    })

# Save metadata
pd.DataFrame(meta).to_csv(os.path.join(OUTDIR, "metadata.csv"), index=False)

print("Done.")
print("Total files created:", len(os.listdir(OUTDIR)))

Done.
Total files created: 301
